# SmartBugs-Wild ML Pipeline — one-click reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/[your-username]/smartbugs-wild-ml/blob/main/notebooks/pipeline_colab.ipynb)

This notebook downloads the data, builds weak labels from nine analysis tools,
trains five models (Random Forest, SVM, CNN, GNN, Ensemble), and reports F1
scores with confidence intervals and McNemar significance tests.

**Before you trust any number, read `README.md` and `docs/METHODOLOGY.md`.**
The labels are tool-consensus *weak labels*, not human ground truth — the models
learn to predict what a committee of imperfect tools would say. See the tutorial
for the full explanation.

**Runtime tip:** for the CNN/GNN, enable a GPU via
*Runtime → Change runtime type → T4 GPU*. CPU also works, just slower.

## Step 1 — Get the code and install dependencies

In [ ]:
!git clone https://github.com/[your-username]/smartbugs-wild-ml.git
%cd smartbugs-wild-ml
!pip install -r requirements.txt -q
print("Environment ready.")

## Step 2 — Download the data

Two downloads: the 32 MB label file (always) and the ~few-hundred-MB contract
corpus. The contract download is the slow part (a few minutes). It is
idempotent — safe to re-run.

In [ ]:
from src import data_download
data_download.download(download_contracts=True)

## Step 3 — Build weak labels and inspect the distribution

This is the most important table in the project. Notice the heavy class
imbalance (`arithmetic` dominates) — it explains why we report macro-F1, not
just accuracy.

In [ ]:
from src import labeling, config

label_df = labeling.build_label_table()      # consensus K = 2 by default
print(labeling.label_summary(label_df).to_string(index=False))

### (Optional) See how the threshold changes everything
Lowering the consensus to K = 1 keeps every single-tool flag: more positives,
more noise. There is no "correct" K — reporting the trade-off is part of doing
this honestly.

In [ ]:
config.CONSENSUS_K = 1
print(labeling.label_summary(labeling.build_label_table()).to_string(index=False))
config.CONSENSUS_K = 2   # restore default

## Step 4 — Configure the run size

`MAX_CONTRACTS` keeps the first run fast. Start small (6,000), confirm it works,
then raise it (or set to `None` for all 47k). Bigger = slower but steadier
numbers.

In [ ]:
config.MAX_CONTRACTS = 6000   # try None once you trust the setup
config.EPOCHS = 8             # neural-net training epochs
print("MAX_CONTRACTS =", config.MAX_CONTRACTS)

## Step 5 — Run the full pipeline

Sample → features → train all five models → evaluate. Watch the CNN/GNN loss
fall epoch by epoch (that's learning) and read the final table. **Those are your
real results** — the numbers to put in your write-up.

In [ ]:
from src import pipeline
results = pipeline.run(max_contracts=config.MAX_CONTRACTS, run_neural=True)

## Step 6 — Read the statistics like a researcher

Per-category performance, then McNemar significance vs. the ensemble. A small
p-value means a real difference; a large one means you can't claim one model
beats another.

In [ ]:
import json
metrics = json.load(open("artifacts/metrics.json"))

print("=== Ensemble per-category ===")
for row in metrics["results"]["Ensemble"]["per_category"]:
    print(f"  {row['category']:20s} F1={row['f1']:.3f}  "
          f"P={row['precision']:.3f}  R={row['recall']:.3f}  n={row['support']}")

print("\n=== McNemar vs Ensemble ===")
for name, m in metrics["mcnemar_vs_ensemble"].items():
    verdict = "significant" if m["p_value"] < 0.05 else "not significant"
    print(f"  {name:13s} p={m['p_value']}  ({verdict})")

## Step 7 — Save your results

Download `metrics.json` and keep it with your paper. Paste the table into your
README so others can see your actual, reproducible numbers.

In [ ]:
from google.colab import files
files.download("artifacts/metrics.json")

---
### Where to go next
- Validate on the 143 hand-labeled **SmartBugs-Curated** contracts to see how
  well "predict the tools" transfers to real ground truth.
- Replace the co-occurrence graph with an AST/CFG graph for the GNN.
- Tune per-class thresholds on the validation split to lift macro-F1.

See `docs/METHODOLOGY.md` and `TUTORIAL.md` for details and the honesty
checklist to follow before publishing.